### テーブル別アクセス数・利用ユーザー数（リソース分析）

`gold_audit_table_daily`

目的：テーブル別アクセス数、利用ユーザー数（リソース分析）
スキーマ例：

```
event_date DATE                          # 監査ログ基準日（日単位）
table_name STRING                        # アクセス対象テーブル名
audit_event_count LONG                   # テーブルへの総アクセス回数
distinct_user_count LONG                 # そのテーブルにアクセスしたユニークユーザー数
get_table_count LONG                     # getTable操作回数
command_submit_count LONG                # commandSubmit操作回数
```

In [0]:
%run ../config

In [0]:
gold_table_name = "gold_audit_table_daily"
gold_table_path = f"{catalog_name}.{schema_name}.{gold_table_name}"


In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {gold_table_path} (
        event_date DATE,
        table_name STRING,
        audit_event_count LONG,
        distinct_user_count LONG,
        get_table_count LONG,
        command_submit_count LONG
    )
    PARTITIONED BY (event_date)
    """
)

GENIEを頼りにシルバーテーブルからゴールドテーブルを作ってみましょう。

プロンプト例
```
`event_id` STRING,
`event_time` TIMESTAMP,
`action_name` STRING,
`resource_name` STRING,
`source_ip` STRING,
`user` STRING,
`user_email` STRING,
`user_name` STRING,
`request_params` STRING,

上記スキーマのシルバーauditテーブルから、spark.sqlで以下スキーマのようなgoldテーブルを作成して

event_date DATE                          # 監査ログ基準日（日単位）
table_name STRING                        # アクセス対象テーブル名
audit_event_count LONG                   # テーブルへの総アクセス回数
distinct_user_count LONG                 # そのテーブルにアクセスしたユニークユーザー数
get_table_count LONG                     # getTable操作回数
command_submit_count LONG                # commandSubmit操作回数
```

※ 本ハンズオンでは Spark SQL を中心に使っていますが、好みに合わせて DataFrame API（withColumnなど）を使っていただいても問題ございません。

In [0]:
# テーブルとして保存
df.write.mode("overwrite").saveAsTable(gold_table_path)

print("✅ gold_daily_stats テーブルを作成しました")

In [0]:
# 結果を確認
display(spark.table(gold_table_path))
